### Import

In [1]:
import numpy as np
import pandas as pd
import datetime as dt

pd.options.display.max_rows = 1000
pd.options.display.max_colwidth = 10000
pd.set_option('display.max_columns', 500)

### general parameters 

In [2]:
mimiciv = "PATH TO DATA/mimiciv(3.1)/"

### hosp - patients

In [3]:
patients = pd.read_csv(mimiciv + "hosp/patients.csv")
patients['dod'] = pd.to_datetime(patients['dod'])

## Select Covid Patients

In [4]:
patients = patients[patients.anchor_year_group == '2020 - 2022'].reset_index(drop=True)

covid_patients_id = list(patients.subject_id.unique())

In [5]:
patients.head(3)

In [6]:
print(patients.subject_id.nunique())

49173


### hosp - admission

In [29]:
admissions = pd.read_csv(mimiciv + "hosp/admissions.csv")

In [30]:
admissions.drop(columns=['admission_type', 'admit_provider_id', 'admission_location', 'discharge_location',
                         'insurance', 'language', 'marital_status', 'edregtime', 'edouttime'], inplace=True)

In [31]:
admissions = admissions[admissions.subject_id.isin(covid_patients_id)].reset_index(drop=True)

In [32]:
admissions.head(3)

In [33]:
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])
admissions['deathtime'] = pd.to_datetime(admissions['deathtime'])

In [34]:
admissions.head(3)

In [36]:
print(admissions.subject_id.nunique())
print(admissions.hadm_id.nunique())

28677
42491


### merge admission and patients

In [37]:
cohort_df = pd.merge(admissions, patients, on=['subject_id'], how='left')

In [38]:
cohort_df['death_time_disch'] = cohort_df['dod'] - cohort_df['dischtime']
cohort_df['death_time_disch'] = pd.to_timedelta(cohort_df['death_time_disch'])
cohort_df['death_time_disch'] = (cohort_df.death_time_disch / pd.Timedelta(days = 1)).round(0)
cohort_df['death_time_disch'] = cohort_df['death_time_disch'] + 1

In [39]:
cohort_df['age'] = (cohort_df['admittime'].dt.year - cohort_df['anchor_year']) + cohort_df['anchor_age']
cohort_df.drop(columns=['dod', 'anchor_year', 'anchor_age'], inplace=True)
cohort_df = cohort_df[['subject_id', 'hadm_id', 'gender', 'age', 'race', 'admittime', 'dischtime', 
                       'deathtime', 'hospital_expire_flag', 'death_time_disch', 'anchor_year_group']]

In [40]:
cohort_df.head(3)

In [41]:
print(cohort_df.subject_id.nunique())
print(cohort_df.hadm_id.nunique())

28677
42491


### icu - icustays

In [42]:
icustays = pd.read_csv(mimiciv + "icu/icustays.csv")

In [43]:
icustays.drop(columns=['first_careunit', 'last_careunit'], inplace=True)

In [44]:
icustays['intime']  = pd.to_datetime(icustays['intime'])
icustays['outtime'] = pd.to_datetime(icustays['outtime'])
icustays['los'] = icustays['los'].round(1)

In [45]:
icustays.head(3)

In [46]:
print(icustays.subject_id.nunique())
print(icustays.hadm_id.nunique())
print(icustays.stay_id.nunique())

65366
85242
94458


### merge cohort and icu

In [47]:
cohort_df = pd.merge(cohort_df, icustays, on=['subject_id', 'hadm_id'], how='left')

In [48]:
cohort_df['ADM_IN_DIFF'] = cohort_df['intime'] - cohort_df['admittime']
cohort_df['ADM_IN_DIFF'] = pd.to_timedelta(cohort_df['ADM_IN_DIFF'])
cohort_df['ADM_IN_DIFF'] = (cohort_df.ADM_IN_DIFF / pd.Timedelta(hours = 1)).round(1)

cohort_df['OUT_DIS_DIFF'] = cohort_df['dischtime'] - cohort_df['outtime']
cohort_df['OUT_DIS_DIFF'] = pd.to_timedelta(cohort_df['OUT_DIS_DIFF'])
cohort_df['OUT_DIS_DIFF'] = (cohort_df.OUT_DIS_DIFF / pd.Timedelta(hours = 1)).round(1)

In [49]:
cohort_df = cohort_df[~(((cohort_df.OUT_DIS_DIFF < -24) & (cohort_df.hospital_expire_flag == 0)))]

condition_1 = (cohort_df.ADM_IN_DIFF < 0)
cohort_df.loc[condition_1, 'admittime']  = cohort_df.loc[condition_1, 'intime']

condition_2 = ((cohort_df.OUT_DIS_DIFF < 0) & (cohort_df.hospital_expire_flag == 0))
cohort_df.loc[condition_2, 'dischtime']  = cohort_df.loc[condition_2, 'outtime']

cohort_df = cohort_df[cohort_df.OUT_DIS_DIFF > -48]

condition_3 = (cohort_df.OUT_DIS_DIFF < 0)
cohort_df.loc[condition_3, 'outtime']  = cohort_df.loc[condition_3, 'dischtime']

In [50]:
cohort_df.drop(columns=['OUT_DIS_DIFF', 'ADM_IN_DIFF'], inplace=True)

In [51]:
cohort_df = cohort_df[(cohort_df.stay_id.notnull()) & (cohort_df.intime >= cohort_df.admittime) & 
                      (cohort_df.outtime <= cohort_df.dischtime)]

In [52]:
cohort_df['icu_expire_flag'] = 0
cohort_df.loc[((cohort_df['deathtime'] > cohort_df['intime']) & (cohort_df['deathtime'] < cohort_df['outtime'] 
                                                              + pd.Timedelta(hours=6))), 'icu_expire_flag'] = 1

In [53]:
cohort_df['icuLos_h'] = cohort_df['outtime'] - cohort_df['intime']
cohort_df['icuLos_h'] = pd.to_timedelta(cohort_df['icuLos_h'])

cohort_df['icuLos_h'] = (cohort_df.icuLos_h / pd.Timedelta(hours = 1)).round(1)
cohort_df.rename(columns={"los": "icuLos_d"}, inplace=True)

In [54]:
cohort_df['hosp_Los'] = cohort_df['dischtime'] - cohort_df['admittime']
cohort_df['hosp_Los'] = pd.to_timedelta(cohort_df['hosp_Los'])

cohort_df['hospLos_d'] =  (cohort_df.hosp_Los / pd.Timedelta(days = 1)).round(1)
cohort_df['hospLos_h'] =  (cohort_df.hosp_Los / pd.Timedelta(hours= 1)).round(1)

cohort_df.drop(columns=['hosp_Los'], inplace=True)

In [55]:
cohort_df = cohort_df[['subject_id', 'hadm_id', 'stay_id', 'gender', 'age', 'race', 'intime', 'outtime',
                       'admittime', 'dischtime', 'icuLos_d', 'icuLos_h', 'hospLos_h', 'hospLos_d',
                       'icu_expire_flag', 'hospital_expire_flag', 'deathtime', 'death_time_disch',
                       'anchor_year_group']]

In [56]:
cohort_df.head(3)

In [57]:
print(cohort_df.subject_id.nunique())
print(cohort_df.hadm_id.nunique())
print(cohort_df.stay_id.nunique())
print(cohort_df[cohort_df.death_time_disch.notnull()].stay_id.nunique())

8563
9380
10564
3187


### Save Covid Data

In [58]:
covid_subject_id = list(cohort_df.subject_id.unique())
covid_hadm_id = list(cohort_df.hadm_id.unique())
covid_stay_id = list(cohort_df.stay_id.unique())

In [60]:
with open("../Data/Cohort/covid_subject_id.txt", "w") as f:
    for subject_id in covid_subject_id:
        f.write(str(subject_id) +"\n")
        
with open("../Data/Cohort/covid_hadm_id.txt", "w") as f:
    for hadm_id in covid_hadm_id:
        f.write(str(hadm_id) +"\n")
        
with open("../Data/Cohort/covid_stay_id.txt", "w") as f:
    for stay_id in covid_stay_id:
        f.write(str(stay_id) +"\n")

### Split for Extraction

In [33]:
def split_list(original_list, num_splits):
    
    split_size = len(original_list) // num_splits
    split_lists = [[] for _ in range(num_splits)]
    
    for i, item in enumerate(original_list):
        index = i // split_size
        if index >= num_splits:
            index = num_splits - 1
        split_lists[index].append(item)

    return split_lists

In [34]:
original_list = list(cohort_df.subject_id.unique())
split_lists = split_list(original_list, 3)

In [35]:
for i, lst in enumerate(split_lists):
    print(f"List {i+1} size: {len(lst)}")

List 1 size: 16662
List 2 size: 16662
List 3 size: 16664


In [36]:
chr1 = split_lists[0]
chr2 = split_lists[1]
chr3 = split_lists[2]

In [37]:
cohort1 = cohort_df[cohort_df.subject_id.isin(chr1)].copy()
cohort2 = cohort_df[cohort_df.subject_id.isin(chr2)].copy()
cohort3 = cohort_df[cohort_df.subject_id.isin(chr3)].copy()

In [38]:
print(cohort1.stay_id.nunique())
print(cohort2.stay_id.nunique())
print(cohort3.stay_id.nunique())

23791
23875
23766


In [39]:
cohort1_subject_id = list(cohort1.subject_id.unique())
cohort1_hadm_id = list(cohort1.hadm_id.unique())
cohort1_stay_id = list(cohort1.stay_id.unique())

cohort2_subject_id = list(cohort2.subject_id.unique())
cohort2_hadm_id = list(cohort2.hadm_id.unique())
cohort2_stay_id = list(cohort2.stay_id.unique())

cohort3_subject_id = list(cohort3.subject_id.unique())
cohort3_hadm_id = list(cohort3.hadm_id.unique())
cohort3_stay_id = list(cohort3.stay_id.unique())

In [40]:
cohort1_stay_id = [int(id) for id in cohort1_stay_id]
cohort2_stay_id = [int(id) for id in cohort2_stay_id]
cohort3_stay_id = [int(id) for id in cohort3_stay_id]

In [41]:
with open("../Data/Cohort/cohort1_subject_id.txt", "w") as f:
    for subject_id in cohort1_subject_id:
        f.write(str(subject_id) +"\n")
        
with open("../Data/Cohort/cohort1_hadm_id.txt", "w") as f:
    for hadm_id in cohort1_hadm_id:
        f.write(str(hadm_id) +"\n")
        
with open("../Data/Cohort/cohort1_stay_id.txt", "w") as f:
    for stay_id in cohort1_stay_id:
        f.write(str(stay_id) +"\n")

In [42]:
with open("../Data/Cohort/cohort2_subject_id.txt", "w") as f:
    for subject_id in cohort2_subject_id:
        f.write(str(subject_id) +"\n")
        
with open("../Data/Cohort/cohort2_hadm_id.txt", "w") as f:
    for hadm_id in cohort2_hadm_id:
        f.write(str(hadm_id) +"\n")
        
with open("../Data/Cohort/cohort2_stay_id.txt", "w") as f:
    for stay_id in cohort2_stay_id:
        f.write(str(stay_id) +"\n")

In [43]:
with open("../Data/Cohort/cohort3_subject_id.txt", "w") as f:
    for subject_id in cohort3_subject_id:
        f.write(str(subject_id) +"\n")
        
with open("../Data/Cohort/cohort3_hadm_id.txt", "w") as f:
    for hadm_id in cohort3_hadm_id:
        f.write(str(hadm_id) +"\n")
        
with open("../Data/Cohort/cohort3_stay_id.txt", "w") as f:
    for stay_id in cohort3_stay_id:
        f.write(str(stay_id) +"\n")